In [1]:
from nd2reader import ND2Reader
import tifffile
import numpy as np
from matplotlib import pyplot as plt
import glob
import sys
import numpy as np
import os
import importlib
from scipy.ndimage import label
import natsort
import shutil
import re

# Inputs

In [2]:
test_set = '/Users/bebr1814/projects/anabaena/scratch_data/bulk_test_set'
# Expect "_img.tiff" and "_mask.tiff" files that have not been used to train the model
seg_results = '/Users/bebr1814/projects/anabaena/scratch_data/bulk_segmentation_output'
# Expect "_segmented.tiff" files with segmentation results from the model (including the test set)


In [ ]:
def evaluate_segmentation(mask,seg):
	# mask = ground truth
	# seg = segmentation result

	assert mask.shape == seg.shape, "Mask and segmentation must have the same shape"

	# Find the pixel-by-pixel overlap between the mask and the segmentation
	# First, make both binary (anything > 0 is 1) so that we can compare them easily (labels won't match up anyway)
	mask = np.where(mask > 0, 1, 0)
	seg = np.where(seg > 0, 1, 0)

	# Find the overlap (logical AND)
	overlap = mask * seg

	# Calculate the Jaccard index
	jaccard = np.sum(overlap) / np.sum(mask + seg - overlap)

	# Calculate the Sørensen-Dice Index
	dice = 2*np.sum(overlap) / (np.sum(mask) + np.sum(seg))

	return jaccard, dice


In [ ]:
metrics = {}

# Grab all the test set files
test_imgs = natsort.natsorted(glob.glob(os.path.join(test_set, '*_img.tiff')))
# test_masks = natsort.natsorted(glob.glob(os.path.join(test_set, '*_mask.tiff')))

# Grab all the segmentation results
seg_imgs = natsort.natsorted(glob.glob(os.path.join(seg_results,'*_segmented.tiff')))

for test_img in test_imgs:
	# For each test image, find the corresponding segmentation result
	label = os.path.basename(test_img).replace('_img.tiff', '')
	test_mask = test_img.replace('_img.tiff', '_mask.tiff')
	assert os.path.exists(test_mask), f"Could not find mask for {test_img}"
	seg = os.path.join(seg_results, label + '_segmented.tiff')

	mask_array = tifffile.imread(test_mask)
	seg_array = tifffile.imread(seg)

	# Run the evaluation
	jaccard, dice = evaluate_segmentation(mask_array, seg_array)

	print(test_img, '- Jaccard:',jaccard, '- Dice:', dice)

	metrics[label] = {'jaccard': jaccard, 'dice': dice}

